In [ ]:
import numpy as np
import matplotlib.pypot as plt

#fixando sementinhas pra manter a reprodutibilidade
np.random.seed(2022)

#gerar 500 amostras siuladas para distribuiçao, 
#definindo a media com 10.0 e o desvio parao de 2.0

nu_real = 10.0
sigma_real = 2.0
x_amostras = np.random.normal(loc = nu_real, scale = sigma_real)

#agora as funçoes manuais pra calcular a media e o desvio
def calcula_media_manual(vetor):
    return np.sum(vetor) / len(vetor)
    #soma os termos e divide pelo array

def calcula_std_manual(vetor, media):
        soma_desvios_quad = np.sum((vetor - media) ** 2)
        return np.sqrt(soma_desvios_quad / (len(vetor) -1))
        #calculando o desvio padrao amostral usando graus de liberdade -1

media_calc = calcular_media_manual(x_amostras)
std_calc = calcula_std_manual(x_amostras, media_calc)
print(F"Media calculada: {media_calc:.4f} (Real: {nu_real})")
print(f"Desvio padrao 06 calculado: {std_calc:.4f} (Real: {sigma_real})")

In [ ]:
#agora é plotagem dos histogramas vs a densidade analitica teorica
def densidade_gaussiana_teorica(x, nu, sigma):
    #a forma matematica exata f(x) da distribuição normal
    termo_escala = 1.0 / (sigma * np.sqrt(2 * np.pi))
    exponente = -0.5 * ((x - nu) / sigma) ** 2
    return termo_escala * np.exp(exponente)

x_intervalo = np.linspace(4,16,200)
y_densidade = densidade_gaussiana_teorica(x_intervalo, nu_real, sigma_real)

plt.figure(figsize=(10,5))
plt.hist(x_amostras, bins=25, density=True, alpha = 0.6, color='blue',edgecolor='black',
          label='Dados Simulados')
plt.plot(x_intervalo, y_densidade, color='red', linewidth=2.5, label=''
'(Densidade teorica f(x))')
plt.title('Distribuição Gaussiana Simulada vs. Curva de Sino Analítica')
plt.xlabel('Valores do atributo')
plt.ylabel('Frequencia de Ocorrencia')
plt.legend()
plt.grid(True, lineStyle='--', alpha=0.5)
plt.show()


In [ ]:
import pandas as pd

#criando o dataframe simulando dados reais ruidosos do titanic
dados_titanic = {
    'PassangerId':[1, 2, 3, 4, 5, 2, 6, 7],
    'Survived': [0,1,1,1,0,1,0,1],
    'Pclass':[3,1,3,1,3,1,2,3],
    'Sex': ['male', 'female','female', 'female', 'male', 'female','male', 'female'],
    'Age': ['22.0', '38.0','26.0', '35,0','null','38.0', '29.0','null'],
    'Fare': [7.25,71.28,7.92,53.10,8.05,71.28,13,00,8.05]
}

df_raw = pd.DataFrame(dados_titanic)
print("DataFrame Original com redundancias e Strings nulas: ")
print(df_raw)


In [ ]:
try:
    #isso vai falhar, pq a coluna Age tem um tipo de texto
   print("Tentando engatar essa porra de caculo na coluna ruidosa: ")
   media_ruidosa = df_raw['Age'].mean()
except TypeError as e:
   print("---- TRACEBACK CONTROLADO ----")
   print(f"TypeError detectado com sucesso: {e}")
   print("Explicativo: O PC nao consegue computar os desvios matematicos sobre strings." \
   "Devemos coagir tipos primeiro")

In [ ]:
df_clean = df_raw.copy()
df_clean = df_clean.drop_duplicates(subset=['PassangerId'], keep='first')
#agora vem a solução estruturada e certinha do dataframe
df_clean['Age'] = pd.to_numeric(df_clean['Age'], errors='coerce')
print("DataFrame Limpinho e tipado: ")
print(df_clean)

In [ ]:
#gerando dados d idade simulados com nulospara diagnostico profundo
np.random.seed(42)
idades_originais = np.random.normal(loc=30, scale=8,size=100)
df_diag = pd.DataFrame({'Age': idades_originais})
df_diag.loc[df_diag.sample(frac=0.2).index, 'Age'] = np.nan
#essafuncao adiciona 20% de valores nulos aos dados
#solucao 1: input variada pela media global
df_media_global = df_diag.copy()
df_media_global['Age'] = df_media_global['Age'].fillna(df_media_global['Age'].mean())
#imprie tudo os bang matematico de variancia~
var_original = df_diag['Age'].var()
var_imputada_media = df_media_global['Age'].var()
print(f"Variancia Amostral oriinal sem os nulos: {var_original:.4f}")
print(f"Variancia amostral pos inputação por media: {var_imputada_media:.4f}")
print(f"Contraçao de variancia calculada de: {((var_original - var_imputada_media)
                                               / var_original)*100:.2f}%")


In [ ]:
#Agora vamos desenhar
plt.figure(figsize=(10, 5))
df_diag['Age'].plot(kind='kde',color='black', style='--',
                    label='(Original sem os nulos):' ,ax=plt.gca())
df_media_global['Age'].plot(kind='kde',color='red',
                label='Inputaçao media global(contraqcao de variancia):' ,ax=plt.gca())
plt.title('Distorcao da distruibuicao de probabilidade por imputação media: ')
plt.xlabel('Idade')
plt.ylabel('Densidade de probabilidade')
plt.legend()
plt.grid(True, lineStyle="--", alpha=0.5)
plt.show()


In [ ]:
#carregar o dataframe com idades nulas e classes de passagem
df_sistema = df_clean.copy()

#agrupando pelo criteio de classeencontrandoa mediana de cada grupo
grupo_mediana = df_sistema.groupby('Pclass')['Age'].transform('median')
df_sistema['Age'] = df_sistema['Age'].fillna(grupo_mediana)
medianas_reais = df_clean.groupby('Pclass')['Age'].median()
print("Medianas calculadas locais dos grupos: ")
print(df_sistema)